<a href="https://colab.research.google.com/github/mntalha/LLM_Atom_Gen/blob/main/colabnotebooks/MistralExampleGeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth
    !pip install jarvis-tools pymatgen gdown

In [1]:
%%time
import os
os.chdir('/content')
!git clone https://github.com/mntalha/LLM_Atom_Gen.git
os.chdir('/content/LLM_Atom_Gen')


Cloning into 'LLM_Atom_Gen'...
remote: Enumerating objects: 464, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 464 (delta 65), reused 43 (delta 14), pack-reused 316 (from 1)
Receiving objects: 100% (464/464), 121.17 MiB | 7.88 MiB/s, done.
Resolving deltas: 100% (236/236), done.
CPU times: user 141 ms, sys: 27 ms, total: 168 ms
Wall time: 20.9 s


In [3]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available.")

GPU is available!
Number of GPUs available: 1
GPU Name: Tesla T4


In [8]:
import numpy as np
from tqdm import tqdm
import os
from models import load_model, get_model, get_trainer, save_model
import os
from unsloth import FastLanguageModel
from datasets import load_dataset
import pandas as pd
from utils import eval_prompts
from sample_funcs import parse_fn
from transformers import TextStreamer
import time
import random
import numpy as np
import random


In [14]:
import gdown
gdown.download_folder("https://drive.google.com/drive/folders/1_gjPU-N7rOz09fS7PVR4hAWyzu-HUp8D?usp=share_link", quiet=True, use_cookies=False)

['/content/LLM_Atom_Gen/mistral-7b-bnb-4bit/adapter_config.json',
 '/content/LLM_Atom_Gen/mistral-7b-bnb-4bit/adapter_model.safetensors',
 '/content/LLM_Atom_Gen/mistral-7b-bnb-4bit/README.md',
 '/content/LLM_Atom_Gen/mistral-7b-bnb-4bit/special_tokens_map.json',
 '/content/LLM_Atom_Gen/mistral-7b-bnb-4bit/tokenizer_config.json',
 '/content/LLM_Atom_Gen/mistral-7b-bnb-4bit/tokenizer.json',
 '/content/LLM_Atom_Gen/mistral-7b-bnb-4bit/tokenizer.model']

In [16]:
# Model configuration
max_seq_length = 2048  # Maximum sequence length for the model
dtype = None           # Data type (None lets the model decide)
load_in_4bit = True    # Use 4-bit quantization for efficiency

# Load the fine-tuned language model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/LLM_Atom_Gen/mistral-7b-bnb-4bit", # Path to your trained model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map="auto"  # Automatically select device (GPU/CPU)
)
# Enable faster inference mode
FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2025.6.2: Fast Mistral patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.6.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): l

In [17]:
prompt_example = "Below is a description of a superconductor material. Write a response that appropriately completes the request.\n\n### Instruction:\nGenerate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n### Input:\nThe chemical formula is MgB2. The  mbj_bandgap value is 6.483.\n\n### Response:\n"


In [18]:
batch = tokenizer(prompt_example, return_tensors="pt")
batch = {k: v.cuda() for k, v in batch.items()}
generate_ids = model.generate(
                                    **batch,
                                    do_sample=False,
                                    max_new_tokens=4096,
                                    pad_token_id=tokenizer.eos_token_id,
                                    use_cache=True)

gen_strs = tokenizer.batch_decode(
                                    generate_ids,
                                    skip_special_tokens=True,
                                    clean_up_tokenization_spaces=True
                                    )

In [19]:
gen_strs

['Below is a description of a superconductor material. Write a response that appropriately completes the request.\n\n### Instruction:\nGenerate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n### Input:\nThe chemical formula is MgB2. The  mbj_bandgap value is 6.483.\n\n### Response:\n3.3 3.3 3.3\n60 60 60\nMg\n0.00 0.00 0.00\nB\n0.75 0.75 0.75\nB\n0.25 0.25 0.25']

In [22]:
material_str = gen_strs[0].replace(prompt_example, "")
material_str

'3.3 3.3 3.3\n60 60 60\nMg\n0.00 0.00 0.00\nB\n0.75 0.75 0.75\nB\n0.25 0.25 0.25'

In [20]:
cif_str = parse_fn(material_str)
cif_str

"# generated using pymatgen\ndata_MgB2\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   3.30000000\n_cell_length_b   3.30000000\n_cell_length_c   3.30000000\n_cell_angle_alpha   60.00000000\n_cell_angle_beta   60.00000000\n_cell_angle_gamma   60.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   MgB2\n_chemical_formula_sum   'Mg1 B2'\n_cell_volume   25.41129640\n_cell_formula_units_Z   1\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Mg  Mg0  1  0.00000000  0.00000000  0.00000000  1\n  B  B1  1  0.75000000  0.75000000  0.75000000  1\n  B  B2  1  0.25000000  0.25000000  0.25000000  1\n"